# OptiGene — Jupyter Notebook Demo
### Mata Kuliah: Parallel Processing (Komputasi Paralel)

Notebook ini mendemonstrasikan secara interaktif seluruh alur kerja proyek **OptiGene**:
1. **Data Layer**: Mengunduh data historis (yfinance) dan web scraping suku bunga (BI & SBN).
2. **Genetic Algorithm**: Mencari kombinasi portofolio optimal berdasarkan batasan risiko.
3. **Komputasi Paralel**: Menjalankan benchmark perbandingan performa 6 metode (Sekuensial Python, PySpark SQL, PySpark RDD map, PySpark RDD filter+reduce, CUDA GPU, dan PySpark + CUDA hybrid) serta visualisasi grafik *speedup* secara inline.

## 1. Verifikasi Lingkungan & Import Modul

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Masukkan folder root workspace ke PATH
sys.path.append(os.path.abspath("."))

from backend.pyspark.session import get_spark_session
from backend.cuda.kernels import CUDA_AVAILABLE
from backend.orchestrator import optimize_portfolio_flow

print("=== OPTIGENE ENVIRONMENT STATUS ===")
print(f"CUDA (GPU via CuPy) Tersedia: {CUDA_AVAILABLE}")

# Inisialisasi Spark secara inline
spark = get_spark_session()
print(f"PySpark Version: {spark.version}")

## 2. Scraping Data & Analisis Historis Aset
Kita mengambil data historis dari yfinance (saham LQ45 & emas) serta scraping data suku bunga BI Rate & SBN ORI terbaru.

In [ ]:
from backend.data.fetcher import fetch_bi_rate, fetch_sbn_rate, LQ45_TICKERS
from backend.data.cache import load_prices_cache

bi_rate = fetch_bi_rate()
sbn_rate = fetch_sbn_rate()

print(f"Suku Bunga Deposito (BI Rate): {bi_rate * 100:.2f}%")
print(f"Suku Bunga SBN (DJPPR Yield 10Y): {sbn_rate * 100:.2f}%")

# Muat cache harga historis
df_prices = load_prices_cache()
if df_prices.empty:
    print("Cache kosong, silakan jalankan optimasi pertama untuk mengunduh data dari yfinance.")
else:
    print(f"\nJumlah Aset Terdaftar: {len(df_prices.columns)} (Emas + Saham LQ45 lolos filter)")
    print("\nSampel Harga Historis (5 data pertama):")
    display(df_prices.head())

## 3. Eksekusi Optimasi Portofolio (Genetic Algorithm)
Kita jalankan optimasi pencarian alokasi dana terbaik menggunakan Genetic Algorithm. Kita coba untuk profil risiko **Aman** dan **Agresif**.

In [ ]:
capital = 10000000.0 # Modal Rp 10.000.000

# 1. Profil Risiko Aman (Maksimal Saham 20%)
print("\n--- RUNNING GA OPTIMIZER: AMAN ---")
res_safe = optimize_portfolio_flow(capital, "aman", duration_years=3, mode="numpy_vectorized")
print(f"Proyeksi Return: {res_safe['return']['percentage']}")
print(f"Tingkat Volatilitas (Risiko): {res_safe['volatility']['percentage']} ({res_safe['volatility']['label']})")
print(f"Sharpe Ratio: {res_safe['sharpe']['value']} ({res_safe['sharpe']['label']})")

# 2. Profil Risiko Agresif (Maksimal Saham 80%)
print("\n--- RUNNING GA OPTIMIZER: AGRESIF ---")
res_agg = optimize_portfolio_flow(capital, "agresif", duration_years=3, mode="numpy_vectorized")
print(f"Proyeksi Return: {res_agg['return']['percentage']}")
print(f"Tingkat Volatilitas (Risiko): {res_agg['volatility']['percentage']} ({res_agg['volatility']['label']})")
print(f"Sharpe Ratio: {res_agg['sharpe']['value']} ({res_agg['sharpe']['label']})")

### Visualisasi Alokasi Dana (Pie Chart)
Mari kita visualisasikan ke mana uang Rp 10.000.000 tersebut harus ditempatkan berdasarkan profil **Aman** vs **Agresif**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Subplot 1: Aman
safe_alloc = res_safe["allocation"]
safe_labels = [item["asset"] for item in safe_alloc]
safe_sizes = [float(item["percentage"].replace("%", "")) for item in safe_alloc]
axes[0].pie(safe_sizes, labels=safe_labels, autopct='%1.1f%%', startangle=140, shadow=True)
axes[0].set_title("Rekomendasi Portofolio: AMAN (Rp 10.000.000)", fontsize=12, fontweight='bold')

# Subplot 2: Agresif
agg_alloc = res_agg["allocation"]
agg_labels = [item["asset"] for item in agg_alloc]
agg_sizes = [float(item["percentage"].replace("%", "")) for item in agg_alloc]
axes[1].pie(agg_sizes, labels=agg_labels, autopct='%1.1f%%', startangle=140, shadow=True)
axes[1].set_title("Rekomendasi Portofolio: AGRESIF (Rp 10.000.000)", fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Benchmark Performa Komputasi Paralel
Bagian inti akademik: Kita membandingkan 6 metode komputasi dalam mengevaluasi Sharpe Ratio 1000 kombinasi portofolio secara berurutan, dan memverifikasi presisi kesamaan hasilnya.

In [ ]:
from backend.benchmark.runner import run_benchmark

# 1. Persiapan matriks parameter
df_returns_dynamic = df_prices.pct_change().dropna()
T_ret = len(df_returns_dynamic)
deposito_daily_ret = bi_rate / 252.0
sbn_daily_ret = sbn_rate / 252.0

df_returns = pd.DataFrame(index=df_returns_dynamic.index)
df_returns["DEPOSITO"] = np.full(T_ret, deposito_daily_ret)
df_returns["SBN ORI"] = np.full(T_ret, sbn_daily_ret)
for col in df_returns_dynamic.columns:
    df_returns[col] = df_returns_dynamic[col]
    
asset_names = df_returns.columns.tolist()
mu = df_returns.mean().values * 252.0
mu[0] = bi_rate
mu[1] = sbn_rate
Sigma = np.cov(df_returns.values, rowvar=False) * 252.0

P_rel = np.zeros((T_ret + 1, len(asset_names)))
P_rel[0, :] = 1.0
for t in range(1, T_ret + 1):
    P_rel[t, 0] = (1.0 + deposito_daily_ret) ** t
    P_rel[t, 1] = (1.0 + sbn_daily_ret) ** t
for col_idx, col_name in enumerate(asset_names[2:], start=2):
    initial_price = df_prices[col_name].iloc[0]
    if initial_price > 0:
        P_rel[:, col_idx] = df_prices[col_name].values[:T_ret+1] / initial_price
    else:
        P_rel[:, col_idx] = 1.0

# 2. Jalankan benchmark 6 metode
df_benchmark = run_benchmark(df_prices, mu, Sigma, P_rel, profile_name="seimbang")
display(df_benchmark)

### Plot Perbandingan Waktu Eksekusi & Speedup
Mari kita visualisasikan akselerasi komputasi GPU & CPU paralel dibanding Sekuensial Python menggunakan diagram batang.

In [ ]:
plt.figure(figsize=(12, 6))
colors = ['#f87171', '#60a5fa', '#fbbf24', '#34d399', '#a78bfa', '#fb923c']

# Bar chart untuk waktu eksekusi
bars = plt.bar(df_benchmark["Method"], df_benchmark["Time (s)"], color=colors[:len(df_benchmark)])
plt.ylabel("Waktu Eksekusi (Detik)", fontsize=11, fontweight='bold')
plt.title("Analisis Performa Komputasi: Waktu Eksekusi Portofolio (P=1.000)", fontsize=13, fontweight='bold')
plt.xticks(rotation=15)

# Berikan anotasi angka di atas bar
for bar in bars:
    yval = bar.get_height()
    if not np.isnan(yval):
        plt.text(bar.get_x() + bar.get_width()/2.0, yval + (max(df_benchmark["Time (s)"])*0.01), f"{yval:.4f} s", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

# Print resume speedup
print("\n=== RESUME AKSELERASI PORTFOLIO EVALUATION ===")
for idx, row in df_benchmark.iterrows():
    print(f"{row['Method']}: {row['Time (s)']:.4f} s (Speedup: {row['Speedup']:.2f}x)")